# **Kalshi NFL Player Prop Market Analysis**

In [283]:
import numpy as np
import pandas as pd
from utils import *
import plotly.express as px
from collections import Counter
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Preprocessing

In [311]:
SERIES_TICKERS = {
    "anytime_td"      : "KXNFLANYTD",     # NFL: 'CMC scores at least one touch down in a game'
    "two_plus_tds"    : "KXNFL2TD",       # NFL: 'CMC scores at least two touch downs in a game'
    "first_td"        : "KXNFLFIRSTTD",   # NFL: 'CMC scores the first touch down in a game'
    "rushing_yards"   : "KXNFLRSHYDS",    # NFL: 'Brock Purdy records 40+ rushing yards in a game'
    "receiving_yards" : "KXNFLRECYDS",    # NFL: 'CMC records 20+ receiving yards in a game'
    "passing_yards"   : "KXNFLPASSYDS",   # NFL: 'Brock Purdy records 200+ passing yards in a game'
    "passing_tds"     : "KXNFLPASSTDS",   # NFL: 'Brock Purdy throws 2+ touch downs in a game'
    "receptions"      : "KXNFLREC",       # NFL: 'CMC records 4+ receptions in a game'
}

### Preprocessing - Market Data

In [285]:
# Load in data

markets_df = pd.read_parquet('data/raw/historical_markets.parquet')

In [286]:
markets_df.head(2)

,can_close_early,close_time,created_time,custom_strike,early_close_condition,event_ticker,expected_expiration_time,expiration_time,expiration_value,last_price_dollars,...,updated_time,volume_24h_fp,volume_fp,yes_ask_dollars,yes_ask_size_fp,yes_bid_dollars,yes_bid_size_fp,yes_sub_title,floor_strike,subtitle
0,True,2026-02-09T03:47:16Z,2026-02-05T18:12:48.735839Z,{'football_player': 'f710b516-32b8-4cc7-ae91-8...,This market will close and expire early if the...,KXNFLANYTD-26FEB08SEANE,2026-02-09T02:30:00Z,2026-02-22T23:30:00Z,No,0.0100,...,2026-02-19T08:48:16.628714Z,0.00,107875.00,1.0000,0.00,0.0000,0.00,George Holani,NaN,None
1,True,2026-02-09T02:37:51Z,2026-02-05T18:12:48.432779Z,{'football_player': '42a1f21f-07d2-41e4-b204-c...,This market will close and expire early if the...,KXNFLANYTD-26FEB08SEANE,2026-02-09T02:30:00Z,2026-02-22T23:30:00Z,Yes,0.9900,...,2026-02-19T08:48:16.628714Z,0.00,182540.00,1.0000,0.00,0.0000,0.00,Mack Hollins,NaN,None


In [287]:
# Drop duplicate tickers

markets_df = markets_df.drop_duplicates(['ticker'])

In [288]:
# Filter for liquid markets

markets_df = markets_df[markets_df['volume_fp'].astype(float) > 0]

In [289]:
# Market is finalized

markets_df = markets_df[markets_df['status'] == 'finalized']

In [290]:
# Remove 'scalar' (ie the game was cancelled)

markets_df = markets_df[markets_df['result'] != 'scalar']

In [291]:
# Create series column

markets_df['series'] = markets_df['ticker'].apply(lambda x: x.split('-')[0])

In [292]:
markets_df.series.value_counts()

series
KXNFLRECYDS      8875
KXNFLANYTD       5230
KXNFLREC         4978
KXNFLFIRSTTD     4736
KXNFLRSHYDS      4374
KXNFLPASSYDS     3135
KXNFL2TD         1953
KXNFLPASSTDS      738
KXNFLGAMESACK      14
Name: count, dtype: int64

In [293]:
# Drop 'sacks' and 'No Touchdown' markets

markets_df = markets_df.loc[(markets_df['series'] != "KXNFLGAMESACK") & (markets_df['yes_sub_title'] != "No Touchdown"), :]

In [294]:
# Clean up index

markets_df = markets_df.reset_index(drop=True)

In [298]:
# Create player column

split_col = markets_df['yes_sub_title'].str.split(':', n=1, expand=True)
markets_df['player']  = split_col[0].str.strip()

In [299]:
# Map duplicate players to formal name

name_map = {
    'Devonta Smith': 'DeVonta Smith', 
    'James Cook III': 'James Cook',
    'Hollywood Brown': 'Marquise Brown', 
    'Bam Knight': 'Zonovan Knight'
}

markets_df['player'] = markets_df['player'].replace(name_map)

markets_df['player'].value_counts()[:10]

player
Drake Maye             281
Josh Allen             256
Caleb Williams         256
Christian McCaffrey    253
Jalen Hurts            241
Kenneth Walker III     238
Sam Darnold            236
Matthew Stafford       233
Kyren Williams         231
Bo Nix                 228
Name: count, dtype: int64

### Preprocessing - Trade Data

In [301]:
# Load in data

trades_df = pd.read_parquet('data/raw/historical_trades.parquet')

In [302]:
trades_df.head(2)

,count_fp,created_time,is_block_trade,no_price_dollars,taker_book_side,taker_outcome_side,taker_side,ticker,trade_id,yes_price_dollars
0,80.00,2026-02-09T03:08:50.42768Z,False,0.9900,bid,yes,yes,KXNFLANYTD-26FEB08SEANE-SEAESAUBERT81,8e43a171-1168-7940-7792-c01925e63cd3,0.0100
1,467.00,2026-02-09T02:57:25.323792Z,False,0.9900,bid,yes,yes,KXNFLANYTD-26FEB08SEANE-SEAESAUBERT81,073a6a01-9432-4957-b99e-bc62cc79ce28,0.0100


In [303]:
trades_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2276812 entries, 0 to 2276811
Data columns (total 10 columns):
 #   Column              Dtype 
---  ------              ----- 
 0   count_fp            object
 1   created_time        object
 2   is_block_trade      bool  
 3   no_price_dollars    object
 4   taker_book_side     object
 5   taker_outcome_side  object
 6   taker_side          object
 7   ticker              object
 8   trade_id            object
 9   yes_price_dollars   object
dtypes: bool(1), object(9)
memory usage: 158.5+ MB


In [304]:
# Convert data

numeric_cols = ['count_fp', 'yes_price_dollars', 'no_price_dollars']

trades_df[numeric_cols] = trades_df[numeric_cols].apply(pd.to_numeric)

trades_df['created_time'] = pd.to_datetime(trades_df['created_time'], format='ISO8601', utc=True).dt.tz_convert('America/New_York')

In [305]:
# Drop duplicate trades

trades_df = trades_df.drop_duplicates(['trade_id']).reset_index(drop=True)

In [306]:
# Add dollar amount column

trades_df['dollar_amt'] = trades_df['count_fp'] * np.where(
    trades_df['taker_outcome_side'] == 'no', 
    trades_df['no_price_dollars'],
    trades_df['yes_price_dollars']
)

## **Left Off Here - Start**

In [308]:
# Create game date, away team, and home team columns

date_teams = trades_df['ticker'].str.split('-', n=1).str[1]

year  = date_teams.str[0:2]      # "26"
month = date_teams.str[2:5]      # "JUN"
day   = date_teams.str[5:7]      # "13"

trades_df['game_date'] = pd.to_datetime(
    year + month + day,
    format='%y%b%d'
)

trades_df['away_team'] = date_teams.str[7:10]
trades_df['home_team'] = date_teams.str[10:13]

trades_df.head(3)

,count_fp,created_time,is_block_trade,no_price_dollars,taker_book_side,taker_outcome_side,taker_side,ticker,trade_id,yes_price_dollars,dollar_amt,game_date,away_team,home_team
0,80.0,2026-02-08 22:08:50.427680-05:00,False,0.99,bid,yes,yes,KXNFLANYTD-26FEB08SEANE-SEAESAUBERT81,8e43a171-1168-7940-7792-c01925e63cd3,0.01,0.80,2026-02-08,SEA,NE-
1,467.0,2026-02-08 21:57:25.323792-05:00,False,0.99,bid,yes,yes,KXNFLANYTD-26FEB08SEANE-SEAESAUBERT81,073a6a01-9432-4957-b99e-bc62cc79ce28,0.01,4.67,2026-02-08,SEA,NE-
2,9.0,2026-02-08 21:55:28.298331-05:00,False,0.99,bid,yes,yes,KXNFLANYTD-26FEB08SEANE-SEAESAUBERT81,f9c5458f-cb92-71d3-3771-a7608a42f918,0.01,0.09,2026-02-08,SEA,NE-


In [321]:
trades_df['home_team'].value_counts()

home_team
NE-    497299
HI-    218725
EA-    173782
PIT    137924
SF-     96618
        ...  
E-L       205
E-N       115
V-N       104
C-N         5
F-N         4
Name: count, Length: 72, dtype: int64

### Preprocessing - NFL Game Start Data

## **Left Off Here - End**

### Preprocessing - Create Prop Data Frames

In [318]:
markets_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33598 entries, 0 to 33597
Data columns (total 47 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   can_close_early           33598 non-null  bool   
 1   close_time                33598 non-null  object 
 2   created_time              33598 non-null  object 
 3   custom_strike             33598 non-null  object 
 4   early_close_condition     33598 non-null  object 
 5   event_ticker              33598 non-null  object 
 6   expected_expiration_time  33598 non-null  object 
 7   expiration_time           33598 non-null  object 
 8   expiration_value          33598 non-null  object 
 9   last_price_dollars        33598 non-null  object 
 10  latest_expiration_time    33598 non-null  object 
 11  liquidity_dollars         33598 non-null  object 
 12  market_type               33598 non-null  object 
 13  no_ask_dollars            33598 non-null  object 
 14  no_bid

In [319]:
trades_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2276812 entries, 0 to 2276811
Data columns (total 14 columns):
 #   Column              Dtype                           
---  ------              -----                           
 0   count_fp            float64                         
 1   created_time        datetime64[ns, America/New_York]
 2   is_block_trade      bool                            
 3   no_price_dollars    float64                         
 4   taker_book_side     object                          
 5   taker_outcome_side  object                          
 6   taker_side          object                          
 7   ticker              object                          
 8   trade_id            object                          
 9   yes_price_dollars   float64                         
 10  dollar_amt          float64                         
 11  game_date           datetime64[ns]                  
 12  away_team           object                          
 13  home_team   

In [ ]:
# Merge data frames

merged = trades_df.merge(markets_df, on='ticker', how='left')

final = merged[[
    'ticker', 'created_time_x', 'count_fp', 'close_time', 'series',
    'game_start_time', 'dollar_amt', 'result', 'yes_price_dollars',
    'player',
]].rename(columns={'created_time_x': 'created_time'}).dropna().reset_index(drop=True).copy()

final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2243661 entries, 0 to 2243660
Data columns (total 9 columns):
 #   Column             Dtype                           
---  ------             -----                           
 0   ticker             object                          
 1   created_time       datetime64[ns, America/New_York]
 2   count_fp           float64                         
 3   close_time         object                          
 4   series             object                          
 5   dollar_amt         float64                         
 6   result             object                          
 7   yes_price_dollars  float64                         
 8   player             object                          
dtypes: datetime64[ns, America/New_York](1), float64(3), object(5)
memory usage: 154.1+ MB


In [314]:
# Create 'prop' data frames

prop_dfs = {
    name: final[final['series'] == ticker].reset_index(drop=True).copy()
    for name, ticker in SERIES_TICKERS.items()
}

anytime_td_df      = prop_dfs['anytime_td']
two_plus_tds_df    = prop_dfs['two_plus_tds']
first_td_df        = prop_dfs['first_td']
rushing_yards_df   = prop_dfs['rushing_yards']
receiving_yards_df = prop_dfs['receiving_yards']
passing_yards_df   = prop_dfs['passing_yards']
passing_tds_df     = prop_dfs['passing_tds']
receptions_df      = prop_dfs['receptions']

In [315]:
print('Dollar Volume by Series:')
final.groupby('series')['dollar_amt'].sum().apply(lambda x: f"${x:,.0f}")

Dollar Volume by Series:


series
KXNFL2TD         $6,063,110
KXNFLANYTD      $44,133,925
KXNFLFIRSTTD     $6,596,253
KXNFLPASSTDS     $2,529,122
KXNFLPASSYDS     $9,759,477
KXNFLREC         $2,873,130
KXNFLRECYDS     $13,041,323
KXNFLRSHYDS     $14,055,213
Name: dollar_amt, dtype: object

In [316]:
print('Markets (tickers) by Series:')
final.groupby('series')['ticker'].count().apply(lambda x: f"{x:,}")

Markets (tickers) by Series:


series
KXNFL2TD        182,394
KXNFLANYTD      984,923
KXNFLFIRSTTD    310,379
KXNFLPASSTDS     62,902
KXNFLPASSYDS    166,637
KXNFLREC         59,134
KXNFLRECYDS     234,807
KXNFLRSHYDS     242,485
Name: ticker, dtype: object

# Analysis

In [1]:
LIQUIDITY_AMOUNT: float = 25.0      # Pre-game liquidity size for simulation
TRADES: int = 5                     # Required number of pre-game trades for simulation
DOLLAR_VOLUME: float = 50.0         # Required dollar volume pre-game for simulation
KALSHI_MAKER_FEE: float = 0.0175    # Kalshi maker fee for June 2026
KALSHI_TAKER_FEE: float = 0.07      # Kalshi taker fee for June 2026